# Polymarket Analysis

This notebook extracts and analyzes Polymarket prediction market data.

## Features:
- Fetches orderbook depth (YES & NO tokens) from CLOB API
- Retrieves price history (1w, 1m)
- Computes market stats and trading signals
- Persists data to DuckDB

## Usage:
Run all cells in order. The last cell will prompt for a Polymarket URL.


## 1. Imports and Configuration


In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Polymarket Data Extractor — Extended Version (with fixes)
---------------------------------------------------------
Features:
1) Accepts a Polymarket Event or Market URL
2) Resolves markets via Gamma API
3) Fetches:
   - Orderbook depth (YES & NO tokens) from CLOB
   - Price history (1w, 1m) from CLOB
4) Computes & stores:
   - Full market stats (your requested fields)
   - Added: best_ask_no, best_bid_no
   - Signals:
       * base_rate_deviation
       * sentiment_momentum (REGRESSION SLOPE of price over last week)
       * liquidity_score
       * degen_risk
5) Persists to DuckDB tables:
   - polymarket_orderbook
   - polymarket_prices_history
   - polymarket_market_stats

Fixes in this version:
- Regression slope uses np.ptp(x) (avoids AttributeError in some envs)
- Orderbook rows reliably get snapshot_ts before MERGE upsert
- More robust category sourcing (market/event)
- Safer MERGE upserts for all three tables
"""

from __future__ import annotations
from typing import Dict, Any, List, Optional, Tuple
from urllib.parse import urlparse
import time
import math
import numpy as np
import requests
import pandas as pd
import duckdb
from datetime import datetime, timezone
import matplotlib.pyplot as plt

# -------------------------
# Config & endpoints
# -------------------------
GAMMA_BASE = "https://gamma-api.polymarket.com"
CLOB_BASE = "https://clob.polymarket.com"

REQUEST_TIMEOUT = 30
MAX_RETRIES = 3
RETRY_BACKOFF_SEC = 2

DUCKDB_PATH = "markets.duckdb"
TBL_OB = "polymarket_orderbook"
TBL_HIST = "polymarket_prices_history"
TBL_STATS = "polymarket_market_stats"

DEFAULT_DEPTH = 10               # per side
DEFAULT_INTERVALS = ["1w", "1m"] # fixed windows
DEFAULT_FIDELITY = 60            # minutes
USE_UTC = True

# Analytics defaults
BASE_RATE = 0.50                 # tune to your priors

# -------------------------
# Utilities
# -------------------------

## 2. Utility Functions


In [2]:
def _utc_now() -> datetime:
    """Return current UTC datetime."""
    return datetime.now(timezone.utc)

def _retry_get(url: str, params: Optional[Dict[str, Any]] = None) -> requests.Response:
    """GET with retry & backoff."""
    last_exc = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            r = requests.get(url, params=params, timeout=REQUEST_TIMEOUT)
            r.raise_for_status()
            return r
        except Exception as e:
            last_exc = e
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_BACKOFF_SEC * attempt)
            else:
                raise
    # If here, last_exc is not None
    raise last_exc


## 3. URL Parsing


In [3]:
def parse_polymarket_url(url: str) -> Dict[str, str]:
    """
    Returns dict: { 'kind': 'event'|'market', 'slug': '<slug>' }
    Supports https://polymarket.com/event/<slug> and /market/<slug>
    """
    p = urlparse(url)
    parts = [s for s in p.path.split("/") if s]
    if not parts:
        raise ValueError("Unrecognized Polymarket URL path")

    if parts[0] in ("event", "events"):
        if len(parts) < 2:
            raise ValueError("Event URL missing slug segment")
        return {"kind": "event", "slug": parts[1]}
    if parts[0] in ("market", "markets"):
        if len(parts) < 2:
            raise ValueError("Market URL missing slug segment")
        return {"kind": "market", "slug": parts[1]}

    # Fallback: treat first segment as slug; prefer event
    return {"kind": "event", "slug": parts[0]}


## 4. Gamma API Resolvers


In [4]:
def get_event_by_slug(slug: str) -> Dict[str, Any]:
    return _retry_get(f"{GAMMA_BASE}/events/slug/{slug}").json()

def get_event_by_id(eid: str | int) -> Dict[str, Any]:
    return _retry_get(f"{GAMMA_BASE}/events/{eid}").json()

def get_market_by_slug(slug: str) -> Dict[str, Any]:
    return _retry_get(f"{GAMMA_BASE}/markets/slug/{slug}").json()

def get_market_by_id(mid: str | int) -> Dict[str, Any]:
    return _retry_get(f"{GAMMA_BASE}/markets/{mid}").json()


def _normalize_clob_token_ids(raw) -> List[str]:
    """Normalize `clobTokenIds` into a list of token-id strings.

    Handles common Gamma encodings:
      - list-like: ["<id1>", "<id2>"]
      - JSON string of a list
      - comma/semicolon-separated string
      - returns [] when not parseable
    """
    if raw is None:
        return []

    # Already a list
    if isinstance(raw, list):
        return [str(x) for x in raw if x is not None]

    # Sometimes stored as a JSON-ish string
    if isinstance(raw, str):
        s = raw.strip()
        if not s:
            return []
        if s.startswith('[') and s.endswith(']'):
            try:
                import json as _json
                arr = _json.loads(s)
                if isinstance(arr, list):
                    return [str(x) for x in arr if x is not None]
            except Exception:
                # fall through to delimiter split
                pass
        # delimiter split
        parts = [t.strip() for t in s.replace(';', ',').split(',') if t.strip()]
        return parts

    # Unknown type
    try:
        return [str(raw)]
    except Exception:
        return []



def get_yes_no_token_ids(market: Dict[str, Any]) -> Tuple[Optional[str], Optional[str], Dict[str, Any]]:
    """Resolve YES/NO token IDs robustly using Gamma `outcomes` when available.

    Returns (yes_token_id, no_token_id, meta).
    """
    meta: Dict[str, Any] = {
        "mapping_source": None,
        "outcomes": None,
        "yes_index": None,
        "no_index": None,
        "mapping_ok": False,
        "mapping_warning": None,
    }

    raw_ids = _normalize_clob_token_ids(market.get("clobTokenIds")) or []
    clob_ids = [t for t in raw_ids if t]

    outcomes = market.get("outcomes")

    # Normalize outcomes into list[str] if possible
    if isinstance(outcomes, str):
        s = outcomes.strip()
        if s.startswith('[') and s.endswith(']'):
            try:
                import json as _json
                outcomes = _json.loads(s)
            except Exception:
                outcomes = None
        else:
            outcomes = None

    if isinstance(outcomes, list) and len(outcomes) >= 2 and len(clob_ids) >= 2:
        meta["mapping_source"] = "outcomes"
        meta["outcomes"] = outcomes
        norm = [str(o).strip().lower() for o in outcomes]
        yi = norm.index("yes") if "yes" in norm else None
        ni = norm.index("no") if "no" in norm else None
        meta["yes_index"], meta["no_index"] = yi, ni
        if yi is not None and ni is not None and yi != ni and yi < len(clob_ids) and ni < len(clob_ids):
            meta["mapping_ok"] = True
            return clob_ids[yi], clob_ids[ni], meta
        meta["mapping_warning"] = "outcomes_present_but_yes_no_not_found_or_misaligned"

    # Fallback assumption
    meta["mapping_source"] = "fallback_first_two"
    yes = clob_ids[0] if len(clob_ids) > 0 else None
    no = clob_ids[1] if len(clob_ids) > 1 else None
    meta["mapping_ok"] = bool(yes and no)
    if not meta["mapping_ok"]:
        meta["mapping_warning"] = "missing_clob_token_ids"
    return yes, no, meta
    if isinstance(raw, list):
        return [str(x) for x in raw][:2]
    if isinstance(raw, str):
        s = raw.strip()
        if s.startswith("[") and s.endswith("]"):
            try:
                import json
                arr = json.loads(s)
                if isinstance(arr, list):
                    return [str(x) for x in arr][:2]
            except Exception:
                pass
        parts = [t for t in s.replace(";", ",").split(",") if t.strip()]
        return [p.strip() for p in parts][:2]
    return []


def resolve_markets_from_url(url: str) -> Tuple[List[Dict[str, Any]], Optional[Dict[str, Any]]]:
    """
    Given a Polymarket event/market URL, return (markets, event_if_available).
    Ensures clobTokenIds are present by fetching market-by-id if needed.
    """
    info = parse_polymarket_url(url)
    kind, slug = info["kind"], info["slug"]

    markets: List[Dict[str, Any]] = []
    event_obj: Optional[Dict[str, Any]] = None

    if kind == "event":
        event_obj = get_event_by_slug(slug)
        raw_markets = [m for m in (event_obj.get("markets") or []) if not m.get("closed", False)]
        for m in raw_markets:
            m_full = m
            clob_ids = _normalize_clob_token_ids(m_full.get("clobTokenIds"))
            if not clob_ids:
                mid = m_full.get("id") or m_full.get("marketId") or m_full.get("conditionId")
                if mid is not None:
                    try:
                        mid_int = int(mid) if str(mid).isdigit() else mid
                        m_full = get_market_by_id(mid_int)
                    except Exception:
                        pass
            markets.append(m_full)
    else:
        # kind == "market"
        try:
            m = get_market_by_slug(slug)
        except Exception:
            if slug.isdigit():
                m = get_market_by_id(slug)
            else:
                raise
        markets.append(m)

        # Try to fetch parent event to enrich category/liquidity
        ev_id = m.get("eventId") or m.get("event_id") or m.get("event") or m.get("eventSlug")
        if ev_id:
            try:
                if isinstance(ev_id, str) and not str(ev_id).isdigit():
                    event_obj = get_event_by_slug(ev_id)
                else:
                    event_obj = get_event_by_id(ev_id)
            except Exception:
                event_obj = None

    return markets, event_obj




## 5. CLOB API Functions (Orderbook & Price History)


In [5]:
def fetch_orderbook(token_id: str, depth: int = DEFAULT_DEPTH) -> Dict[str, Any]:
    try:
        r = _retry_get(f"{CLOB_BASE}/book", params={"token_id": token_id})
    except Exception:
        # Market resolved/closed — CLOB returns 404, return empty orderbook
        return {"bids": [], "asks": [], "market": token_id}
    data = r.json() if r.content else {}

    bids = data.get("bids") or []
    asks = data.get("asks") or []

    def _norm(entries, side: str):
        out = []
        for i, e in enumerate(entries[:depth]):
            price = e.get("price", e.get("px"))
            size  = e.get("size",  e.get("qty"))
            if price is None or size is None:
                continue
            out.append({
                "token_id": token_id,
                "side": side,
                "level": i + 1,
                "price": float(price),
                "size": float(size),
            })
        return out

    # Keep book-level metadata so we can match UI logic
    return {
        "bids": _norm(bids, "bid"),
        "asks": _norm(asks, "ask"),
        "last_trade_price": float(data["last_trade_price"]) if (data.get("last_trade_price") not in (None, '')) else None,
        "tick_size": float(data["tick_size"]) if (data.get("tick_size") not in (None, '')) else None,
        "min_order_size": float(data["min_order_size"]) if (data.get("min_order_size") not in (None, '')) else None,
    }

def fetch_prices_history(token_id: str, interval: str, fidelity_min: int = DEFAULT_FIDELITY) -> List[Dict[str, Any]]:
    """
    Return list of rows {token_id, t (UTC), interval, fidelity_min, price}
    """
    r = _retry_get(
        f"{CLOB_BASE}/prices-history",
        params={"market": token_id, "interval": interval, "fidelity": fidelity_min},
    )
    payload = r.json() if r.content else {}
    rows = []
    for z in payload.get("history", []):
        ts = z.get("t")
        price = z.get("p")
        if ts is None or price is None:
            continue
        try:
            dt = datetime.fromtimestamp(int(ts), tz=timezone.utc)
            rows.append(
                {
                    "token_id": token_id,
                    "t": dt,
                    "interval": interval,
                    "fidelity_min": int(fidelity_min),
                    "price": float(price),
                }
            )
        except Exception:
            continue
    return rows


## 6. DuckDB Schema Definitions


In [6]:
DDL_ORDERBOOK = f"""
CREATE TABLE IF NOT EXISTS {TBL_OB} (
    token_id TEXT,
    snapshot_ts TIMESTAMP,   -- snapshot timestamp in UTC
    side TEXT,               -- 'bid' or 'ask'
    level INTEGER,           -- 1..N
    price DOUBLE,
    size DOUBLE,
    PRIMARY KEY (token_id, snapshot_ts, side, level)
);
"""

DDL_HISTORY = f"""
CREATE TABLE IF NOT EXISTS {TBL_HIST} (
    token_id TEXT,
    t TIMESTAMP,       -- bar timestamp (UTC)
    interval TEXT,     -- '1w' or '1m'
    fidelity_min INTEGER, -- minutes
    price DOUBLE,
    PRIMARY KEY (token_id, t, interval, fidelity_min)
);
"""

# =====================================================================
#  UPDATED polymarket_market_stats (FULL FEATURE SET)
# =====================================================================
DDL_STATS = f"""
CREATE TABLE IF NOT EXISTS {TBL_STATS} (
    market_id TEXT,
    snapshot_ts TIMESTAMP,

    title TEXT,
    category TEXT,

    yes_token_id TEXT,
    no_token_id TEXT,

    yes_price DOUBLE,
    no_price DOUBLE,

    -- Explicit pricing fields (token-specific)
    yes_midpoint DOUBLE,
    no_midpoint DOUBLE,
    yes_last_trade DOUBLE,
    no_last_trade DOUBLE,
    yes_display_price DOUBLE,
    no_display_price DOUBLE,

    -- UI-compatible display prices (may apply complementing on non-negRisk markets)
    ui_yes_price DOUBLE,
    ui_no_price DOUBLE,

    -- Token mapping / data-quality flags
    token_mapping_source TEXT,
    token_mapping_ok BOOLEAN,
    token_mapping_warning TEXT,
    token_mapping_anomaly BOOLEAN,
    clob_last_trade_anomaly BOOLEAN,


    best_ask_yes DOUBLE,
    best_bid_yes DOUBLE,
    best_ask_no DOUBLE,
    best_bid_no DOUBLE,

    last_trade_price DOUBLE,

    volume DOUBLE,
    volume_clob DOUBLE,
    volume_1wk DOUBLE,
    volume_1mo DOUBLE,
    liquidity DOUBLE,
    liquidity_clob DOUBLE,

    spread DOUBLE,
    order_min_size DOUBLE,
    min_tick DOUBLE,

    price_change_1d DOUBLE,
    price_change_1wk DOUBLE,
    price_change_1mo DOUBLE,
    price_change_1yr DOUBLE,

    start_date TIMESTAMP,
    end_date TIMESTAMP,
    accepting_orders_since TIMESTAMP,

    active BOOLEAN,
    closed BOOLEAN,
    funded BOOLEAN,
    ready BOOLEAN,

    neg_risk BOOLEAN,
    neg_risk_other BOOLEAN,
    uma_resolution_status TEXT,
    automatically_resolved BOOLEAN,

    created_at TIMESTAMP,
    updated_at TIMESTAMP,

    -- ====== EXTENDED ANALYTICS ======
    volatility_1w DOUBLE,
    ma_short DOUBLE,
    ma_long DOUBLE,
    ema_slope DOUBLE,
    overreaction_flag BOOLEAN,

    orderbook_imbalance DOUBLE,
    slippage_notional_1k DOUBLE,
    slippage_notional_10k DOUBLE,

    fair_value DOUBLE,
    expected_value DOUBLE,
    kelly_fraction DOUBLE,
    trade_signal TEXT,
    late_overconfidence BOOLEAN,

    -- ====== CUSTOM METRICS ======
    base_rate DOUBLE,
    base_rate_deviation DOUBLE,
    sentiment_momentum DOUBLE,
    liquidity_score DOUBLE,
    degen_risk DOUBLE,

    PRIMARY KEY (market_id, snapshot_ts)
);
"""

def _patch_stats_table_for_new_columns(con: duckdb.DuckDBPyConnection):
    """
    One-time (idempotent) in-place migration: add any missing columns
    that the current code expects but older DBs may not have yet.
    """
    expected_cols = {
        # Explicit pricing fields (token-specific)
        "yes_midpoint": "DOUBLE",
        "no_midpoint": "DOUBLE",
        "yes_last_trade": "DOUBLE",
        "no_last_trade": "DOUBLE",
        "yes_display_price": "DOUBLE",
        "no_display_price": "DOUBLE",
        "ui_yes_price": "DOUBLE",
        "ui_no_price": "DOUBLE",
        # Token mapping / data-quality flags
        "token_mapping_source": "TEXT",
        "token_mapping_ok": "BOOLEAN",
        "token_mapping_warning": "TEXT",
        "token_mapping_anomaly": "BOOLEAN",
        "clob_last_trade_anomaly": "BOOLEAN",

        # Extended analytics
        "volatility_1w": "DOUBLE",
        "ma_short": "DOUBLE",
        "ma_long": "DOUBLE",
        "ema_slope": "DOUBLE",
        "overreaction_flag": "BOOLEAN",
        "orderbook_imbalance": "DOUBLE",
        "slippage_notional_1k": "DOUBLE",
        "slippage_notional_10k": "DOUBLE",
        "fair_value": "DOUBLE",
        "expected_value": "DOUBLE",
        "kelly_fraction": "DOUBLE",
        "trade_signal": "TEXT",
        "late_overconfidence": "BOOLEAN",
        # Custom metrics (some may already exist)
        "base_rate": "DOUBLE",
        "base_rate_deviation": "DOUBLE",
        "sentiment_momentum": "DOUBLE",
        "liquidity_score": "DOUBLE",
        "degen_risk": "DOUBLE",
    }

    # Read current columns
    cols_df = con.execute(f"PRAGMA table_info({TBL_STATS});").fetchdf()
    have = set(col.lower() for col in cols_df["name"].tolist())

    # Add missing columns
    for col, sql_type in expected_cols.items():
        if col.lower() not in have:
            con.execute(f"ALTER TABLE {TBL_STATS} ADD COLUMN {col} {sql_type};")

def ensure_tables(con: duckdb.DuckDBPyConnection):
    """Create all tables if not exist, then patch stats to add any missing columns."""
    con.execute(DDL_ORDERBOOK)
    con.execute(DDL_HISTORY)
    con.execute(DDL_STATS)
    _patch_stats_table_for_new_columns(con)


## 7. Database Upsert Functions


In [7]:
def upsert_orderbook(con: duckdb.DuckDBPyConnection, ob_rows: List[Dict[str, Any]], asof: datetime):
    """MERGE upsert for orderbook; adds snapshot_ts column."""
    if not ob_rows:
        return
    df = pd.DataFrame(ob_rows)
    if df.empty:
        return
    df["snapshot_ts"] = asof

    con.register("ob_src", df)
    ensure_tables(con)
    con.execute(f"""
        MERGE INTO {TBL_OB} AS t
        USING (
            SELECT token_id, snapshot_ts, side, level, price, size
            FROM ob_src
        ) AS s
        ON  t.token_id    = s.token_id
        AND t.snapshot_ts = s.snapshot_ts
        AND t.side        = s.side
        AND t.level       = s.level
        WHEN MATCHED THEN UPDATE SET
            price = s.price,
            size  = s.size
        WHEN NOT MATCHED THEN INSERT (
            token_id, snapshot_ts, side, level, price, size
        ) VALUES (
            s.token_id, s.snapshot_ts, s.side, s.level, s.price, s.size
        );
    """)


# =====================================================================
# UPSERT: HISTORY
# =====================================================================
def upsert_history(con: duckdb.DuckDBPyConnection, hist_rows: List[Dict[str, Any]]):
    """MERGE upsert for history bars."""
    if not hist_rows:
        return
    df = pd.DataFrame(hist_rows)
    if df.empty:
        return
    con.register("hist_src", df)
    ensure_tables(con)
    con.execute(f"""
        MERGE INTO {TBL_HIST} AS t
        USING (
            SELECT token_id, t, interval, fidelity_min, price
            FROM hist_src
        ) AS s
        ON t.token_id = s.token_id
       AND t.t = s.t
       AND t.interval = s.interval
       AND t.fidelity_min = s.fidelity_min
        WHEN MATCHED THEN UPDATE SET
            price = s.price
        WHEN NOT MATCHED THEN INSERT (token_id, t, interval, fidelity_min, price)
        VALUES (s.token_id, s.t, s.interval, s.fidelity_min, s.price);
    """)


# =====================================================================
# UPSERT: MARKET STATS  (FINAL FULL VERSION)
# =====================================================================
def upsert_market_stats(con: duckdb.DuckDBPyConnection, stats_rows: List[Dict[str, Any]]):
    """MERGE upsert for market-level stats & signals."""
    if not stats_rows:
        return
    df = pd.DataFrame(stats_rows)
    if df.empty:
        return

    # Normalize timestamps where possible
    for col in ["start_date", "end_date", "accepting_orders_since", "created_at", "updated_at", "snapshot_ts"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce", utc=True)

    con.register("stats_src", df)
    ensure_tables(con)  # ensures and patches the schema *before* merge

    con.execute(f"""
        MERGE INTO {TBL_STATS} AS t
        USING (
            SELECT * FROM stats_src
        ) AS s
        ON  t.market_id    = s.market_id
        AND t.snapshot_ts  = s.snapshot_ts

        WHEN MATCHED THEN UPDATE SET
            title = s.title,
            category = s.category,
            yes_token_id = s.yes_token_id,
            no_token_id = s.no_token_id,
            yes_price = s.yes_price,
            no_price = s.no_price,
            yes_midpoint = s.yes_midpoint,
            no_midpoint = s.no_midpoint,
            yes_last_trade = s.yes_last_trade,
            no_last_trade = s.no_last_trade,
            yes_display_price = s.yes_display_price,
            no_display_price = s.no_display_price,
            ui_yes_price = s.ui_yes_price,
            ui_no_price = s.ui_no_price,
            token_mapping_source = s.token_mapping_source,
            token_mapping_ok = s.token_mapping_ok,
            token_mapping_warning = s.token_mapping_warning,
            token_mapping_anomaly = s.token_mapping_anomaly,
            clob_last_trade_anomaly = s.clob_last_trade_anomaly,
            best_ask_yes = s.best_ask_yes,
            best_bid_yes = s.best_bid_yes,
            best_ask_no = s.best_ask_no,
            best_bid_no = s.best_bid_no,
            last_trade_price = s.last_trade_price,
            volume = s.volume,
            volume_clob = s.volume_clob,
            volume_1wk = s.volume_1wk,
            volume_1mo = s.volume_1mo,
            liquidity = s.liquidity,
            liquidity_clob = s.liquidity_clob,
            spread = s.spread,
            order_min_size = s.order_min_size,
            min_tick = s.min_tick,
            price_change_1d = s.price_change_1d,
            price_change_1wk = s.price_change_1wk,
            price_change_1mo = s.price_change_1mo,
            price_change_1yr = s.price_change_1yr,
            start_date = s.start_date,
            end_date = s.end_date,
            accepting_orders_since = s.accepting_orders_since,
            active = s.active,
            closed = s.closed,
            funded = s.funded,
            ready = s.ready,
            neg_risk = s.neg_risk,
            neg_risk_other = s.neg_risk_other,
            uma_resolution_status = s.uma_resolution_status,
            automatically_resolved = s.automatically_resolved,
            created_at = s.created_at,
            updated_at = s.updated_at,

            -- EXTENDED ANALYTICS
            volatility_1w = s.volatility_1w,
            ma_short = s.ma_short,
            ma_long = s.ma_long,
            ema_slope = s.ema_slope,
            overreaction_flag = s.overreaction_flag,
            orderbook_imbalance = s.orderbook_imbalance,
            slippage_notional_1k = s.slippage_notional_1k,
            slippage_notional_10k = s.slippage_notional_10k,
            fair_value = s.fair_value,
            expected_value = s.expected_value,
            kelly_fraction = s.kelly_fraction,
            trade_signal = s.trade_signal,
            late_overconfidence = s.late_overconfidence,

            -- CUSTOM METRICS
            base_rate = s.base_rate,
            base_rate_deviation = s.base_rate_deviation,
            sentiment_momentum = s.sentiment_momentum,
            liquidity_score = s.liquidity_score,
            degen_risk = s.degen_risk

        WHEN NOT MATCHED THEN INSERT (
            market_id, snapshot_ts,
            title, category,
            yes_token_id, no_token_id,
            yes_price, no_price,
            yes_midpoint, no_midpoint,
            yes_last_trade, no_last_trade,
            yes_display_price, no_display_price,
            ui_yes_price, ui_no_price,
            token_mapping_source, token_mapping_ok, token_mapping_warning, token_mapping_anomaly,
            clob_last_trade_anomaly,
            best_ask_yes, best_bid_yes,
            best_ask_no, best_bid_no,
            last_trade_price,
            volume, volume_clob, volume_1wk, volume_1mo, liquidity, liquidity_clob,
            spread, order_min_size, min_tick,
            price_change_1d, price_change_1wk, price_change_1mo, price_change_1yr,
            start_date, end_date, accepting_orders_since,
            active, closed, funded, ready,
            neg_risk, neg_risk_other, uma_resolution_status, automatically_resolved,
            created_at, updated_at,
            volatility_1w, ma_short, ma_long, ema_slope,
            overreaction_flag, orderbook_imbalance,
            slippage_notional_1k, slippage_notional_10k,
            fair_value, expected_value, kelly_fraction, trade_signal, late_overconfidence,
            base_rate, base_rate_deviation, sentiment_momentum, liquidity_score, degen_risk
        ) VALUES (
            s.market_id, s.snapshot_ts,
            s.title, s.category,
            s.yes_token_id, s.no_token_id,
            s.yes_price, s.no_price,
            s.yes_midpoint, s.no_midpoint,
            s.yes_last_trade, s.no_last_trade,
            s.yes_display_price, s.no_display_price,
            s.ui_yes_price, s.ui_no_price,
            s.token_mapping_source, s.token_mapping_ok, s.token_mapping_warning, s.token_mapping_anomaly,
            s.clob_last_trade_anomaly,
            s.best_ask_yes, s.best_bid_yes,
            s.best_ask_no, s.best_bid_no,
            s.last_trade_price,
            s.volume, s.volume_clob, s.volume_1wk, s.volume_1mo, s.liquidity, s.liquidity_clob,
            s.spread, s.order_min_size, s.min_tick,
            s.price_change_1d, s.price_change_1wk, s.price_change_1mo, s.price_change_1yr,
            s.start_date, s.end_date, s.accepting_orders_since,
            s.active, s.closed, s.funded, s.ready,
            s.neg_risk, s.neg_risk_other, s.uma_resolution_status, s.automatically_resolved,
            s.created_at, s.updated_at,
            s.volatility_1w, s.ma_short, s.ma_long, s.ema_slope,
            s.overreaction_flag, s.orderbook_imbalance,
            s.slippage_notional_1k, s.slippage_notional_10k,
            s.fair_value, s.expected_value, s.kelly_fraction, s.trade_signal, s.late_overconfidence,
            s.base_rate, s.base_rate_deviation, s.sentiment_momentum, s.liquidity_score, s.degen_risk
        );
    """)

## 8. Analytics Helper Functions


In [8]:
def _first_or_none(seq):
    return seq[0] if seq else None

def _get_category(market: Dict[str, Any], event: Optional[Dict[str, Any]]) -> Optional[str]:
    """Find a category from market or event (supports 'category' or 'categories')."""
    for obj in (market, event or {}):
        if not obj:
            continue
        if obj.get("category"):
            return obj.get("category")
        cats = obj.get("categories")
        if isinstance(cats, list) and cats:
            return cats[0]
        if isinstance(cats, str) and cats.strip():
            return cats.strip()
    return None

def _best_ask(ob: Optional[Dict[str, Any]]) -> Optional[float]:
    asks = ob.get("asks") if ob else None
    return float(asks[0]["price"]) if asks else None

def _best_bid(ob: Optional[Dict[str, Any]]) -> Optional[float]:
    bids = ob.get("bids") if ob else None
    return float(bids[0]["price"]) if bids else None

def _display_price(ob: Optional[Dict[str, Any]]) -> Optional[float]:
    if not ob:
        return None
    bb, ba = _best_bid(ob), _best_ask(ob)
    last = ob.get("last_trade_price")
    if bb is not None and ba is not None:
        spread = float(ba) - float(bb)
        # UI fallback when spread > $0.10
        if spread > 0.10 and last is not None:
            return float(last)
        return (float(bb) + float(ba)) / 2.0
    # If one side missing, fall back to the other or last trade
    return (bb or ba or last)

def _compute_depth_liquidity(ob: Optional[Dict[str, Any]], levels: int = 5) -> float:
    """Rough notional liquidity proxy over top N levels for both sides: sum(price * size)."""
    if not ob:
        return 0.0
    total = 0.0
    for side in ("bids", "asks"):
        for e in (ob.get(side) or [])[:levels]:
            try:
                p = float(e.get("price", 0.0))
                q = float(e.get("size", 0.0))
                total += p * q
            except Exception:
                continue
    return total
    
def _round_to_tick(x: Optional[float], tick: Optional[float]) -> Optional[float]:
    if x is None or tick is None or tick <= 0:
        return x
    # round to nearest tick, then to 4 d.p. to avoid float noise
    return round(round(x / tick) * tick, 4)
    
def _latest_hist_price(hist: List[Dict[str, Any]]) -> Optional[float]:
    if not hist:
        return None
    hist_sorted = sorted(hist, key=lambda x: x["t"])
    return float(hist_sorted[-1]["price"])

def regression_slope(history: List[Dict[str, Any]]) -> float:
    """
    (Option A) Price-based regression slope over last week.
    OLS slope of price ~ time; normalized by the time span to stabilize scale.
    Safe against degenerate cases; returns 0.0 when insufficient variation.
    """
    if not history or len(history) < 3:
        return 0.0

    df = pd.DataFrame(history).sort_values("t").dropna(subset=["t", "price"])
    if df.empty or df["t"].nunique() < 2:
        return 0.0

    # Convert to epoch seconds (int64)
    # NOTE: astype('int64') on datetime64[ns, UTC] → nanoseconds; divide to seconds.
    ts_ns = df["t"].astype("int64")
    # If timezone-aware pandas, above is fine; else coerce:
    if (ts_ns == ts_ns.iloc[0]).all():
        # All equal timestamps
        return 0.0

    x = (ts_ns // 10**9).to_numpy(dtype="int64")
    y = df["price"].to_numpy(dtype="float64")

    # Remove any NaNs/inf just in case
    mask = np.isfinite(x) & np.isfinite(y)
    x, y = x[mask], y[mask]
    if x.size < 3 or np.unique(x).size < 2:
        return 0.0

    # Center (improves numerical stability)
    x_c = x - x.mean()
    y_c = y - y.mean()

    var_x = np.var(x_c)
    if var_x == 0:
        return 0.0

    slope = np.cov(x_c, y_c, bias=True)[0, 1] / var_x  # OLS slope

    # Normalize by total time span to make comparable across markets
    span = float(np.ptp(x))  # <-- fix: use np.ptp(array)
    if span <= 0:
        return 0.0

    slope_normalized = float(slope) / span
    return slope_normalized



## 9. Advanced Analytics Functions


In [9]:
def compute_volatility(series):
    if series is None or len(series) < 30:
        return None
    r = series.pct_change().dropna()
    if len(r) < 20:
        return None
    return float(r.rolling(20).std().iloc[-1])

def compute_moving_averages(series):
    if series is None or len(series) < 96:
        return None, None
    ma_short = float(series.rolling(24).mean().iloc[-1])
    ma_long  = float(series.rolling(96).mean().iloc[-1])
    return ma_short, ma_long

def compute_ema_slope(series):
    if series is None or len(series) < 25:
        return None
    ema = series.ewm(span=48, adjust=False).mean()
    tail = ema.iloc[-20:]
    x = np.arange(len(tail))
    y = tail.values
    x_c = x - x.mean()
    y_c = y - y.mean()
    var_x = (x_c**2).mean()
    if var_x == 0:
        return None
    slope = (x_c * y_c).mean() / var_x
    return float(slope)

def detect_overreaction(series, z=2.5):
    if series is None or len(series) < 60:
        return False
    r = series.pct_change().dropna()
    mu = r.rolling(40).mean().iloc[-1]
    std = r.rolling(40).std().iloc[-1]
    if std is None or std == 0:
        return False
    zscore = (r.iloc[-1] - mu) / std
    return abs(zscore) >= z

def compute_orderbook_imbalance(ob):
    if not ob:
        return None
    b = ob.get("bids", [])[:5]
    a = ob.get("asks", [])[:5]
    bq = sum(x["size"] for x in b)
    aq = sum(x["size"] for x in a)
    total = bq + aq
    if total == 0:
        return None
    return float((bq - aq) / total)

def compute_slippage(ob, notional):
    if not ob:
        return None
    best_bids = ob.get("bids", [])
    best_asks = ob.get("asks", [])
    if not best_bids or not best_asks:
        return None

    mid = (best_bids[0]["price"] + best_asks[0]["price"]) / 2

    asks = best_asks
    remaining = notional
    spent = 0
    shares = 0
    for lvl in asks:
        px = lvl["price"]
        qty = lvl["size"]
        cap = px * qty
        take = min(remaining, cap)
        sh = take / px
        spent += sh * px
        shares += sh
        remaining -= take
        if remaining <= 0:
            break
    if shares == 0:
        return None
    avg = spent / shares
    return float((avg - mid) / mid * 10000)

def compute_fair_value(yes_price, base_rate, momentum, vol):
    if yes_price is None:
        return base_rate
    shrink = 1 / (1 + 10*(vol or 0))
    tilt = math.tanh((momentum or 0)*1e5)
    fv = shrink*base_rate + (1-shrink)*yes_price + 0.02*tilt
    return float(min(max(fv,0.01),0.99))

def compute_ev(fv, px):
    if fv is None or px is None:
        return None
    return float(fv - px)

def compute_kelly(fv, px):
    if fv is None or px is None or px <= 0 or px >= 1:
        return None
    p = fv
    q = 1 - p
    b = (1/px) - 1
    if b <= 0:
        return None
    f = (b*p - q)/b
    return float(max(0,min(1,f))) * 0.5

def compute_trade_signal(ev, vol):
    if ev is None:
        return "no-trade"
    ev_bp = ev * 10000
    if ev_bp > 10 and (vol is None or vol < 0.05):
        return "long"
    if ev_bp < -10 and (vol is None or vol < 0.05):
        return "short"
    return "no-trade"

def detect_late_overconfidence(yes, imbalance, best_bid_no):
    if yes is None:
        return False
    if yes < 0.90:
        return False
    if imbalance and imbalance > 0.50:
        return True
    if best_bid_no is not None and best_bid_no < 0.05:
        return True
    return False


## 10. Market Statistics Assembly


In [10]:
def assemble_market_stats(
    market: Dict[str, Any],
    event: Optional[Dict[str, Any]],
    ob_map: Dict[str, Dict[str, Any]],
    hist_map: Dict[Tuple[str, str], List[Dict[str, Any]]],
    asof: datetime,
    base_rate: float = BASE_RATE,
) -> Dict[str, Any]:
    """
    Build a single market stats row using Polymarket's display conventions:
      - Display YES as midpoint unless spread > $0.10, then use last trade.
      - Derive NO = 1 - YES for binary markets to match UI.
      - Round displayed prices to tick size when available.
    """

    # -----------------------------
    # Helpers (local) for display
    # -----------------------------
    def _best_ask(ob: Optional[Dict[str, Any]]) -> Optional[float]:
        asks = ob.get("asks") if ob else None
        return float(asks[0]["price"]) if asks else None

    def _best_bid(ob: Optional[Dict[str, Any]]) -> Optional[float]:
        bids = ob.get("bids") if ob else None
        return float(bids[0]["price"]) if bids else None

    def _display_price_from_ob(ob: Optional[Dict[str, Any]], last_trade_fallback: Optional[float]) -> Optional[float]:
        """
        Apply Polymarket display rule:
          - Use midpoint unless spread > $0.10 → then use last trade price.
          - If only one side exists, use that or last_trade_fallback.
        """
        if not ob:
            return last_trade_fallback
        bb, ba = _best_bid(ob), _best_ask(ob)
        # Prefer the caller's fallback (usually derived from token-specific price
        # history). This avoids a known edge case where CLOB `/book` returns a
        # market-level last_trade_price that can be identical across tokens
        # (observed on `neg_risk` markets).
        last = last_trade_fallback
        if last is None:
            last = ob.get("last_trade_price", None)

        if bb is not None and ba is not None:
            spread = float(ba) - float(bb)
            if spread > 0.10 and last is not None:
                return float(last)
            return (float(bb) + float(ba)) / 2.0
        # Fallback path if one/both sides are missing
        return last or bb or ba

    def _round_to_tick(x: Optional[float], tick: Optional[float]) -> Optional[float]:
        if x is None:
            return None
        if tick is None or tick <= 0:
            # Default to cents if tick is unavailable
            tick = 0.01
        return round(round(float(x) / tick) * tick, 4)



    # -----------------------------
    # Identity & metadata
    # -----------------------------
    market_id = market.get("id") or market.get("marketId") or market.get("conditionId")
    title = market.get("question") or market.get("title")
    category = _get_category(market, event)
    yes_token_id, no_token_id, mapping_meta = get_yes_no_token_ids(market)

    ob_yes = ob_map.get(yes_token_id) if yes_token_id else None
    ob_no  = ob_map.get(no_token_id)  if no_token_id  else None

    # -----------------------------
    # Best quotes, spread, tick
    # -----------------------------
    yes_best_ask = _best_ask(ob_yes)
    yes_best_bid = _best_bid(ob_yes)
    no_best_ask  = _best_ask(ob_no)
    no_best_bid  = _best_bid(ob_no)

    yes_midpoint = (yes_best_bid + yes_best_ask) / 2.0 if (yes_best_bid is not None and yes_best_ask is not None) else None
    no_midpoint  = (no_best_bid  + no_best_ask)  / 2.0 if (no_best_bid  is not None and no_best_ask  is not None) else None

    # Prefer market-level spread if provided; else compute from YES book
    spread = market.get("spread")
    if spread is None and (yes_best_ask is not None) and (yes_best_bid is not None):
        spread = max(0.0, float(yes_best_ask) - float(yes_best_bid))

    # Determine tick size & min order size (prefer /book, fallback Gamma)
    tick_size = (ob_yes or {}).get("tick_size") or market.get("orderPriceMinTickSize") or 0.01
    min_order_size = (ob_yes or {}).get("min_order_size") or market.get("orderMinSize")

    # -----------------------------
    # History & last trade
    # -----------------------------
    # Use history (1w -> 1m) as fallback last trade reference
    hist_yes_1w = hist_map.get((yes_token_id, "1w"), []) if yes_token_id else []
    hist_yes_1m = hist_map.get((yes_token_id, "1m"), []) if yes_token_id else []
    last_trade_from_hist = _latest_hist_price(hist_yes_1w) or _latest_hist_price(hist_yes_1m)

    # Prefer book's last trade for display logic; fall back to history
    # Prefer history-derived last trade (token-specific) over CLOB `/book`
    # last_trade_price (can be market-level on some markets, e.g. neg_risk).
    last_trade_price = last_trade_from_hist or (ob_yes or {}).get("last_trade_price")

    # -----------------------------
    # Display prices (YES / NO)
    # -----------------------------
    neg_risk = bool(market.get("negRisk", False))

    # Diagnostic: On some markets (notably negRisk), CLOB `/book` may return
    # a market-level last_trade_price that is identical for both outcome tokens.
    clob_last_trade_anomaly = False
    if yes_token_id and no_token_id:
        lt_yes = (ob_yes or {}).get("last_trade_price")
        lt_no  = (ob_no  or {}).get("last_trade_price")
        if neg_risk and (lt_yes is not None) and (lt_no is not None) and float(lt_yes) == float(lt_no):
            clob_last_trade_anomaly = True
    
    # 1) Compute YES raw display (mid or last per spread rule)
    yes_display = _display_price_from_ob(ob_yes, last_trade_fallback=last_trade_price)
    
    # 2) Compute NO raw display independently from NO book (TEMP value only)
    #    We'll only *use* this if the market isn't a simple binary (or neg_risk).
    hist_no_1w = hist_map.get((no_token_id, "1w"), []) if no_token_id else []
    hist_no_1m = hist_map.get((no_token_id, "1m"), []) if no_token_id else []
    last_trade_no_hist = _latest_hist_price(hist_no_1w) or _latest_hist_price(hist_no_1m)
    # Same logic for NO token
    last_trade_no = last_trade_no_hist or (ob_no or {}).get("last_trade_price")
    no_display = _display_price_from_ob(ob_no, last_trade_fallback=last_trade_no)
    
    # 3) Round + complement logic (the only place we enforce complement)
    # Keep token-specific display prices separate from UI-compatible complemented prices.
    is_binary = (yes_token_id is not None) and (no_token_id is not None)

    yes_display_price = _round_to_tick(yes_display, tick_size)
    no_display_price  = _round_to_tick(no_display, tick_size)

    ui_yes_price = yes_display_price
    if is_binary and (ui_yes_price is not None) and (not neg_risk):
        # Binary market: derive NO from YES *after* rounding (UI convention)
        ui_no_price = _round_to_tick(1.0 - ui_yes_price, tick_size)
    else:
        # Non-binary or negRisk: keep token-specific NO
        ui_no_price = no_display_price

    # Preserve old variable names for downstream computations
    yes_display = ui_yes_price
    # (no_rounded removed) keep canonical UI-compatible NO price
    no_display = ui_no_price

    # -----------------------------
    # YES series for analytics
    # -----------------------------
    yes_series = None
    if hist_yes_1w:
        df_tmp = pd.DataFrame(hist_yes_1w).sort_values("t")
        yes_series = df_tmp["price"].astype(float)
        yes_series.index = pd.to_datetime(df_tmp["t"], utc=True)

    # -----------------------------
    # Analytics
    # -----------------------------
    volatility = compute_volatility(yes_series)
    ma_short, ma_long = compute_moving_averages(yes_series)
    ema_sl = compute_ema_slope(yes_series)
    overreaction = detect_overreaction(yes_series)
    imbalance = compute_orderbook_imbalance(ob_yes)
    slip_1k = compute_slippage(ob_yes, 1000)
    slip_10k = compute_slippage(ob_yes, 10000)

    fair_value = compute_fair_value(yes_display, base_rate, ema_sl, volatility)
    ev = compute_ev(fair_value, yes_display)
    kelly = compute_kelly(fair_value, yes_display)
    signal = compute_trade_signal(ev, volatility)
    overconf = detect_late_overconfidence(yes_display, imbalance, no_best_bid)

    # your custom metrics
    slope = regression_slope(hist_yes_1w) if hist_yes_1w else 0.0
    depth_liq = _compute_depth_liquidity(ob_yes)
    ev_liq = (event or {}).get("liquidity", 0)
    ev_liq_clob = (event or {}).get("liquidityClob", 0)

    base_rate_deviation = (yes_display - base_rate) if yes_display is not None else None
    liq_score = math.log1p(max(depth_liq + float(ev_liq or 0) + float(ev_liq_clob or 0), 0)) / (1+float(spread or 0))
    spread_norm = min(max(float(spread or 0), 0), 0.5)/0.5
    mom_norm = min(abs(slope)*10, 1)
    liq_inv = 1 - (liq_score / (1+liq_score))
    degen_risk = 0.45*spread_norm + 0.35*mom_norm + 0.20*liq_inv

    # -----------------------------
    # FINAL return object
    # -----------------------------
    return {
        "market_id": str(market_id),
        "snapshot_ts": asof,
        "title": title,
        "category": category,

        "yes_token_id": yes_token_id,
        "no_token_id": no_token_id,

        "clob_last_trade_anomaly": clob_last_trade_anomaly,

        "yes_price": ui_yes_price,
        "no_price": ui_no_price,

        "best_ask_yes": yes_best_ask,
        "best_bid_yes": yes_best_bid,
        "best_ask_no": no_best_ask,
        "best_bid_no": no_best_bid,
        # Explicit price fields (avoid YES/NO confusion)
        "yes_midpoint": _round_to_tick(yes_midpoint, tick_size),
        "no_midpoint": _round_to_tick(no_midpoint, tick_size),
        "yes_last_trade": _round_to_tick(last_trade_from_hist, tick_size),
        "no_last_trade": _round_to_tick(last_trade_no_hist, tick_size),
        "yes_display_price": yes_display_price,
        "no_display_price": no_display_price,
        "ui_yes_price": ui_yes_price,
        "ui_no_price": ui_no_price,

        # Token mapping diagnostics
        "token_mapping_source": mapping_meta.get("mapping_source"),
        "token_mapping_ok": bool(mapping_meta.get("mapping_ok")),
        "token_yes_index": mapping_meta.get("yes_index"),
        "token_no_index": mapping_meta.get("no_index"),
        "token_mapping_warning": mapping_meta.get("mapping_warning"),
        "token_mapping_anomaly": bool(mapping_meta.get("outcomes")) and (not bool(mapping_meta.get("mapping_ok"))),
        "outcomes": mapping_meta.get("outcomes"),

        "last_trade_price": last_trade_price,

        "volume": market.get("volumeNum",0),
        "volume_clob": market.get("volumeClob",0),
        "volume_1wk": market.get("volume1wk",0),
        "volume_1mo": market.get("volume1mo",0),
        "liquidity": ev_liq,
        "liquidity_clob": ev_liq_clob,

        "spread": spread,
        "order_min_size": min_order_size,
        "min_tick": tick_size,

        "price_change_1d": market.get("oneDayPriceChange"),
        "price_change_1wk": market.get("oneWeekPriceChange"),
        "price_change_1mo": market.get("oneMonthPriceChange"),
        "price_change_1yr": market.get("oneYearPriceChange"),

        "start_date": market.get("startDateIso"),
        "end_date": market.get("endDateIso"),
        "accepting_orders_since": market.get("acceptingOrdersTimestamp"),

        "active": market.get("active",False),
        "closed": market.get("closed",False),
        "funded": market.get("funded"),
        "ready": market.get("ready"),

        "neg_risk": neg_risk,
        "neg_risk_other": market.get("negRiskOther"),
        "uma_resolution_status": market.get("umaResolutionStatus"),
        "automatically_resolved": market.get("automaticallyResolved"),

        "created_at": market.get("createdAt"),
        "updated_at": market.get("updatedAt"),

        "volatility_1w": volatility,
        "ma_short": ma_short,
        "ma_long": ma_long,
        "ema_slope": ema_sl,
        "overreaction_flag": bool(overreaction),
        "orderbook_imbalance": imbalance,
        "slippage_notional_1k": slip_1k,
        "slippage_notional_10k": slip_10k,
        "fair_value": fair_value,
        "expected_value": ev,
        "kelly_fraction": kelly,
        "trade_signal": signal,
        "late_overconfidence": overconf,

        "base_rate": base_rate,
        "base_rate_deviation": base_rate_deviation,
        "sentiment_momentum": slope,
        "liquidity_score": liq_score,
        "degen_risk": degen_risk
    }


## 11. Main Extraction Function


In [11]:
def extract_from_url(
    url: str,
    depth: int = DEFAULT_DEPTH,
    intervals: List[str] = DEFAULT_INTERVALS,
    fidelity_min: int = DEFAULT_FIDELITY,
    duckdb_path: str = DUCKDB_PATH,
    base_rate: float = BASE_RATE,
    return_df: bool = False,   # ADD THIS
) -> Dict[str, Any] | pd.DataFrame:

    """
    One-shot extraction for a single URL:
    - resolves markets (+ parent event when possible)
    - for each market:
        * obtains clob token ids (YES & NO)
        * fetches orderbook (depth per side)
        * fetches history for given intervals
        * builds market-level stats (includes signals)
    - persists to DuckDB (orderbook + history + market_stats)
    - returns a summary dict
    """
    markets, event_obj = resolve_markets_from_url(url)
    asof = _utc_now() if USE_UTC else datetime.now()

    all_ob_rows: List[Dict[str, Any]] = []
    all_hist_rows: List[Dict[str, Any]] = []
    all_stats_rows: List[Dict[str, Any]] = []

    for m in markets:
        market_id = m.get("id") or m.get("marketId") or m.get("conditionId")
        clob_ids = _normalize_clob_token_ids(m.get("clobTokenIds"))

        # If still missing, try a direct lookup by id
        if not clob_ids and market_id is not None:
            try:
                mid_int = int(market_id) if str(market_id).isdigit() else market_id
                m_full = get_market_by_id(mid_int)
                clob_ids = _normalize_clob_token_ids(m_full.get("clobTokenIds"))
                m = m_full
            except Exception:
                pass
        # Resolve tokens robustly using outcomes when available
        yes_token_id, no_token_id, _map_meta = get_yes_no_token_ids(m)

        # Expect two tokens: YES, NO
        tokens: List[str] = [t for t in [yes_token_id, no_token_id] if t]

        # Fetch OB + history for each token
        ob_map: Dict[str, Dict[str, Any]] = {}
        hist_map: Dict[Tuple[str, str], List[Dict[str, Any]]] = {}

        for token_id in tokens:
            ob = fetch_orderbook(token_id, depth=depth)
            ob_map[token_id] = ob

            # accumulate orderbook rows
            all_ob_rows.extend(ob["bids"])
            all_ob_rows.extend(ob["asks"])

            # history per interval
            for iv in intervals:
                hist = fetch_prices_history(token_id, interval=iv, fidelity_min=fidelity_min)
                hist_map[(token_id, iv)] = hist
                all_hist_rows.extend(hist)

        # Assemble stats row (includes signals & best_ask_no)
        stats_row = assemble_market_stats(
            market=m,
            event=event_obj,
            ob_map=ob_map,
            hist_map=hist_map,
            asof=asof,
            base_rate=base_rate,
        )
        all_stats_rows.append(stats_row)

    # Persist
    con = duckdb.connect(duckdb_path)
    ensure_tables(con)
    upsert_orderbook(con, all_ob_rows, asof=asof)
    upsert_history(con, all_hist_rows)
    upsert_market_stats(con, all_stats_rows)
    con.close()

    # Small previews (non-truncated)
    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", 1800)

    print("\nOrderbook preview (first 12 rows):\n", pd.DataFrame(all_ob_rows).head(12))
    print("\nHistory preview (first 12 rows):\n", pd.DataFrame(all_hist_rows).head(12))
    print("\nMarket stats + signals:\n", pd.DataFrame(all_stats_rows))

    df = pd.DataFrame(all_stats_rows)
    
    if return_df:
        return df
    
    return {
        "asof": asof,
        "markets": [{"market_id": str(m.get('id') or m.get('marketId') or m.get('conditionId')),
                     "title": m.get('question') or m.get('title')} for m in markets],
        "orderbook_rows": len(all_ob_rows),
        "history_rows": len(all_hist_rows),
        "stats_rows": len(all_stats_rows),
    }

## 12. Run Analysis

**Paste your Polymarket URL when prompted:**


In [12]:
if __name__ == "__main__":
    url = input("Paste a Polymarket Event or Market URL: ").strip()

    df = extract_from_url(url, return_df=True)

    print("\n=== Head ===")
    print(df.head())

    print("\n=== Describe ===")
    print(df.describe(include="all"))

Paste a Polymarket Event or Market URL:  https://polymarket.com/event/bitcoin-above-on-february-23



Orderbook preview (first 12 rows):
                                              token_id side  level  price      size
0   1077987112000504641850773199173943395894962185...  bid      1  0.001  10186.13
1   1077987112000504641850773199173943395894962185...  bid      2  0.002   2250.00
2   1077987112000504641850773199173943395894962185...  bid      3  0.003   2172.64
3   1077987112000504641850773199173943395894962185...  bid      4  0.004    340.00
4   1077987112000504641850773199173943395894962185...  bid      5  0.006     83.00
5   1077987112000504641850773199173943395894962185...  bid      6  0.007      9.73
6   1077987112000504641850773199173943395894962185...  bid      7  0.010  38243.25
7   1077987112000504641850773199173943395894962185...  bid      8  0.020    100.00
8   1077987112000504641850773199173943395894962185...  bid      9  0.021      7.94
9   1077987112000504641850773199173943395894962185...  bid     10  0.030  12250.00
10  4020161045820360342720434914989812701578564467

In [13]:
df.head()

,market_id,snapshot_ts,title,category,yes_token_id,no_token_id,clob_last_trade_anomaly,yes_price,no_price,best_ask_yes,best_bid_yes,best_ask_no,best_bid_no,yes_midpoint,no_midpoint,yes_last_trade,no_last_trade,yes_display_price,no_display_price,ui_yes_price,ui_no_price,token_mapping_source,token_mapping_ok,token_yes_index,token_no_index,token_mapping_warning,token_mapping_anomaly,outcomes,last_trade_price,volume,volume_clob,volume_1wk,volume_1mo,liquidity,liquidity_clob,spread,order_min_size,min_tick,price_change_1d,price_change_1wk,price_change_1mo,price_change_1yr,start_date,end_date,accepting_orders_since,active,closed,funded,ready,neg_risk,neg_risk_other,uma_resolution_status,automatically_resolved,created_at,updated_at,volatility_1w,ma_short,ma_long,ema_slope,overreaction_flag,orderbook_imbalance,slippage_notional_1k,slippage_notional_10k,fair_value,expected_value,kelly_fraction,trade_signal,late_overconfidence,base_rate,base_rate_deviation,sentiment_momentum,liquidity_score,degen_risk
0,1385950,2026-02-23 14:17:51.424719+00:00,"Will the price of Bitcoin be above $58,000 on ...",None,1077987112000504641850773199173943395894962185...,4020161045820360342720434914989812701578564467...,False,1.000,0.000,NaN,0.001,0.999,NaN,NaN,NaN,1.000,0.000,1.000,0.000,1.000,0.000,outcomes,True,0,1,None,False,"[Yes, No]",0.9995,494468.083849,494468.083849,331482.882704,331482.882704,666253.7197,666253.7197,0.001,5.0,0.001,0.0015,None,None,None,2026-02-16,2026-02-23,2026-02-16T17:03:53Z,True,False,False,False,False,False,None,None,2026-02-16T17:00:10.518894Z,2026-02-23T14:06:03.380569Z,0.002424,0.997604,0.992073,0.000100,False,1.000000,NaN,NaN,0.531832,-0.468168,NaN,short,True,0.5,0.500,6.343868e-08,14.088503,0.014155
1,1385954,2026-02-23 14:17:51.424719+00:00,"Will the price of Bitcoin be above $60,000 on ...",None,6671282758995526254666649368362763217282063040...,6207821412583094107798581070900487367373231713...,False,0.998,0.002,0.999,0.001,0.999,0.001,0.5,0.5,0.998,0.002,0.998,0.002,0.998,0.002,outcomes,True,0,1,None,False,"[Yes, No]",0.9985,394329.746458,394329.746458,172024.960859,172024.960859,666253.7197,666253.7197,0.001,5.0,0.001,0.0020,None,None,None,2026-02-16,2026-02-23,2026-02-16T17:03:59Z,True,False,False,False,False,False,None,None,2026-02-16T17:00:10.698391Z,2026-02-23T14:06:08.010355Z,0.006470,0.993604,0.987073,0.000059,False,0.189455,9980.0,9980.0,0.550262,-0.447738,0.0,short,True,0.5,0.498,1.240243e-07,14.096128,0.014149
2,1385956,2026-02-23 14:17:51.424719+00:00,"Will the price of Bitcoin be above $62,000 on ...",None,1671327786883697185705807939091061002581962404...,5196159818462515224112791616751034718645732115...,False,0.992,0.008,0.999,0.001,0.999,0.001,0.5,0.5,0.992,0.008,0.992,0.008,0.992,0.008,outcomes,True,0,1,None,False,"[Yes, No]",0.9925,164711.391066,164711.391066,101133.872451,101133.872451,666253.7197,666253.7197,0.003,5.0,0.001,0.0025,None,None,None,2026-02-16,2026-02-23,2026-02-16T17:04:11Z,True,False,False,False,False,False,None,None,2026-02-16T17:00:11.025898Z,2026-02-23T14:06:10.014484Z,0.034506,0.976771,0.968849,-0.000341,False,-0.659726,9980.0,9980.0,0.606217,-0.385783,0.0,short,True,0.5,0.492,2.508626e-07,14.108319,0.015939
3,1385959,2026-02-23 14:17:51.424719+00:00,"Will the price of Bitcoin be above $64,000 on ...",None,1105705413709230426401137946918983678041931649...,7134482849610557807088101977229990560102741807...,False,0.969,0.031,0.999,0.001,0.999,0.001,0.5,0.5,0.969,0.031,0.969,0.031,0.969,0.031,outcomes,True,0,1,None,False,"[Yes, No]",0.9690,252441.520756,252441.520756,182308.321891,182308.321891,666253.7197,666253.7197,0.002,5.0,0.001,-0.0055,None,None,None,2026-02-16,2026-02-23,2026-02-16T17:04:11Z,True,False,False,False,False,False,None,None,2026-02-16T17:00:11.186366Z,2026-02-23T14:06:09.651993Z,0.110509,0.895229,0.915609,-0.002879,False,-0.063351,9980.0,9980.0,0.726207,-0.242793,0.0,no-trade,True,0.5,0.469,4.238899e-07,14.115558,0.015033
4,1385960,2026-02-23 14:17:51.424719+00:00,"Will

CatalogException: Catalog Error: Table with name polymarket_price_history does not exist!
Did you mean "polymarket_prices_history"?

LINE 2:     SELECT * FROM polymarket_price_history
                          ^